# Day 11 · Senses: Sound — ที่ที่มืดที่สุด คือที่ที่เงียบที่สุดไหม

**#30DayMapChallenge 2026 · วันที่ 11 พ.ย. 2569** · โจทย์: *"How do you map a soundscape? Noise pollution mapped?"*

แกนที่สองของซีรีส์: ระยะจาก hex แต่ละเซลล์ถึง **แหล่งเสียงที่ใกล้ที่สุด** (ถนนสายหลัก · รถไฟ ·
พื้นที่เมือง · สนามบิน จาก Natural Earth 10m) เทียบกับ `km_nearest` (ระยะถึงจุดทะเบียนฟ้ามืด)

**ผล: Spearman −0.021 (p = 0.39)** — ไม่มีความสัมพันธ์ และ 172 จาก 309 hex ที่ "มืด"
อยู่ใกล้แหล่งเสียงไม่ถึง 10 กม.

> บน Colab: สคริปต์จะดาวน์โหลด layer จาก Natural Earth เอง (ถนน 9 MB · รถไฟ 15 MB · เมือง 13 MB · สนามบิน 0.3 MB)


### ตั้งค่า + helper (guards, haversine, distance)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 11 · Senses: Sound — "How do you map a soundscape? Noise pollution mapped?"

คำถามของซีรีส์ต่อจาก Day 10: ที่ที่มืดที่สุด คือที่ที่เงียบที่สุดไหม
เพราะทั้งแสงและเสียงรบกวนมีต้นตอเดียวกันคือคน

วิธี: คำนวณ "ระยะจาก hex แต่ละเซลล์ถึงแหล่งเสียงที่ใกล้ที่สุด" (ถนนสายหลัก · รถไฟ ·
พื้นที่เมือง · สนามบิน จาก Natural Earth 10m) แล้วเทียบกับ km_nearest (ระยะถึงจุด
อนุรักษ์ท้องฟ้ามืดในทะเบียน) — ทั้งสองเป็น proxy ไม่ใช่ค่าวัดจริง และผมจะบอกอย่างนั้นบนแผ่น
"""
import os, sys, json, time, urllib.request, zipfile
import numpy as np, pandas as pd, geopandas as gpd
import shapely
from shapely.geometry import shape, Point, box
from shapely.ops import unary_union
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection
from scipy.stats import spearmanr

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from guards import guard_distances, guard_bbox, guard_crs, guard_area, guard_stat_labels, guard_variance, GuardError

T0 = time.time()
WORK = r"D:\GIS30Days"; REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day11"); OLD = os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day11")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
for f in sorted(os.listdir(OLD)):
    if f.endswith(".ttf"): font_manager.fontManager.addfont(os.path.join(OLD, f))
plt.rcParams["font.family"] = "Trirong"
log = lambda m: print(f"[{time.time()-T0:5.1f}s] {m}", flush=True)

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
AOI = (96, 4, 108, 22)
BG, INK = "#0d1522", "#e8e3d3"
def get(stem, sub="cultural"):
    shp = os.path.join(CACHE, stem + ".shp")
    if not os.path.exists(shp):
        zp = os.path.join(CACHE, stem + ".zip")
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{sub}/{stem}.zip", zp)
        zipfile.ZipFile(zp).extractall(CACHE)
    return shp

### 1. ข้อมูล

In [ ]:
g3 = json.load(open(os.path.join(REPO, "posts/day03-polygons/day03_hex_polygons.geojson"), encoding="utf-8"))
hexes = [shape(f["geometry"]) for f in g3["features"]]
km_dark = np.array([f["properties"]["km_nearest"] for f in g3["features"]], float)
cen = gpd.GeoSeries(hexes, crs="EPSG:4326").to_crs(EQ)
cent = np.array([[g.centroid.x, g.centroid.y] for g in cen])            # ศูนย์กลาง hex ในหน่วย *เมตร* (ใช้วัดระยะ)
cent_deg = np.array([[h.centroid.x, h.centroid.y] for h in hexes])      # ศูนย์กลางใน degree (ใช้ตรวจ bbox)
log(f"hex {len(hexes)} เซลล์ · ระยะถึงจุดทะเบียน mean {km_dark.mean():.1f} กม.")

roads = gpd.read_file(get("ne_10m_roads"), bbox=AOI)
rail  = gpd.read_file(get("ne_10m_railroads"), bbox=AOI)
urban = gpd.read_file(get("ne_10m_urban_areas"), bbox=AOI)
airp  = gpd.read_file(get("ne_10m_airports"), bbox=AOI)
# NE 10m ไม่จำแนกชั้นถนนในไทย (type = Unknown 1,537 / Road 238 / Major Highway 39)
# จึงใช้ถนนทุกสายเป็นแกนหลัก แล้วตรวจความไวด้วย "Major Highway" เท่านั้น
road_all = roads
road_major = roads[roads["type"].astype(str) == "Major Highway"]
log(f"แหล่งเสียง: ถนนทุกสาย {len(road_all)} (Major Highway {len(road_major)}) · รถไฟ {len(rail)} · "
    f"พื้นที่เมือง {len(urban)} · สนามบิน {len(airp)}")

def hex_rings(series):
    for g in series:
        for poly in (g.geoms if g.geom_type.startswith("Multi") else [g]):
            yield np.asarray(poly.exterior.coords)

def dist_km(geoms_wgs):
    """ระยะ (กม.) จากศูนย์กลาง hex แต่ละเซลล์ไปยัง layer ใด layer หนึ่ง
       รวม layer เป็น geometry เดียวก่อน (unary_union) แล้ววัด elementwise —
       หลีกเลี่ยงกับดัก index alignment ของ GeoSeries.distance ที่คืนค่าผิดรูปเงียบ ๆ"""
    merged = unary_union(list(gpd.GeoSeries(geoms_wgs, crs="EPSG:4326").to_crs(EQ).values))
    pts = np.array([Point(x, y) for x, y in cent], dtype=object)
    return shapely.distance(pts, merged) / 1000.0

d_road  = dist_km(road_all.geometry)
d_road_major = dist_km(road_major.geometry)
d_rail  = dist_km(rail.geometry)
d_urban = dist_km(urban.geometry)
d_air   = dist_km(airp.geometry)
noise_km = np.vstack([d_road, d_rail, d_urban, d_air]).min(axis=0)
which = np.vstack([d_road, d_rail, d_urban, d_air]).argmin(axis=0)
noise_major = np.vstack([d_road_major, d_rail, d_urban, d_air]).min(axis=0)
NAMES = ["ถนนสายหลัก", "รถไฟ", "พื้นที่เมือง", "สนามบิน"]
guard_distances(np.vstack([d_road, d_rail, d_urban, d_air, d_road_major]), "day11", max_km=2500)
guard_bbox(cent_deg, 96.5, 107.6, 4.3, 21.7, "hex-centroid-degree")
guard_crs(cent.bounds if hasattr(cent,"bounds") else (cent[:,0].min(), cent[:,1].min(), cent[:,0].max(), cent[:,1].max()), "hex-centroid", expect="metres")
guard_variance(noise_km, "noise_km", min_spread_km=3.0)
guard_variance(d_rail, "d_rail", min_spread_km=1.0)
log(f"ระยะถึงแหล่งเสียงที่ใกล้ที่สุด: mean {noise_km.mean():.1f} กม. · median {np.median(noise_km):.1f} · "
    f"min {noise_km.min():.2f} · max {noise_km.max():.1f}")

### 2. สี่กลุ่ม มืด×เงียบ

In [ ]:
DARK_CUT, LOUD_CUT = 25.0, 5.0
is_dark = km_dark <= DARK_CUT
is_quiet = noise_km >= 10.0
quad = np.select([is_dark & is_quiet, is_dark & ~is_quiet, ~is_dark & is_quiet],
                 ["DQ", "DL", "BQ"], default="BL")
cnt = {q: int((quad == q).sum()) for q in ["DQ", "DL", "BQ", "BL"]}
rho, pv = spearmanr(km_dark, noise_km)
rho_m, pv_m = spearmanr(km_dark, noise_major)
log(f"สี่กลุ่ม: {cnt} · Spearman(ระยะถึงจุดทะเบียน, ระยะถึงแหล่งเสียง) = {rho:+.3f} (p={pv:.1e}) · "
    f"ใช้เฉพาะ Major Highway = {rho_m:+.3f}")

### 3. ฝั่งจุดทะเบียน 83 แห่ง

In [ ]:
df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"))
sp = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326").to_crs(EQ)
def site_dist(layer):
    merged = unary_union(list(gpd.GeoSeries(layer.geometry.values, crs="EPSG:4326").to_crs(EQ).values))
    return shapely.distance(sp.geometry.values, merged) / 1000.0
_sd = np.vstack([site_dist(road_all), site_dist(rail), site_dist(urban), site_dist(airp)])
site_noise = _sd.min(axis=0)
site_which = _sd.argmin(axis=0)
site_src = {NAMES[i]: int((site_which == i).sum()) for i in range(4)}
guard_distances(np.vstack([site_noise]), "sites", max_km=2500)
loud_sites = int((site_noise < LOUD_CUT).sum())
log(f"จุดทะเบียน 83 แห่ง: {loud_sites} แห่งอยู่ใกล้แหล่งเสียง < {LOUD_CUT:.0f} กม. (เสียงถึง) "
    f"(median {np.median(site_noise):.1f} กม.)")

### 4. verification + กันตัวเองก่อนเขียนโพสต์

In [ ]:
assert len(quad) == len(km_dark) and set(np.unique(quad)) == {"DQ", "DL", "BQ", "BL"}
assert np.isfinite(noise_km).all() and noise_km.min() >= 0
assert abs(rho) <= 1.0 and len(cnt) == 4 and sum(cnt.values()) == len(km_dark)
assert int((~is_dark).sum()) == len(km_dark) - int(is_dark.sum())
log("ตรวจแล้ว: สี่กลุ่มครบไม่ซ้ำ · ไม่มี NaN · ผลรวมกลุ่ม = จำนวน hex · Spearman ในช่วง [-1,1]")

### 5. แผนที่

In [ ]:
th = gpd.read_file(os.path.join(OLD, "ne_10m_admin_0_countries.shp"))
TH = gpd.GeoSeries([th[th.ISO_A3 == "THA"].geometry.iloc[0]], crs="EPSG:4326").to_crs(EQ)[0]
COL = {"DQ": "#2dd4bf", "DL": "#f97316", "BQ": "#1e3a5f", "BL": "#f4566a"}
LAB = {"DQ": "มืด + เงียบ", "DL": "มืด แต่มีเสียงรบกวน", "BQ": "สว่าง + เงียบ", "BL": "สว่าง + มีเสียง"}
fig, ax = plt.subplots(figsize=(10.5, 12.2), facecolor=BG); ax.set_facecolor(BG)
ax.set_aspect("equal"); ax.axis("off")
ax.add_collection(PolyCollection([np.asarray(p.exterior.coords) for p in
                                 (TH.geoms if TH.geom_type.startswith("Multi") else [TH])],
                                 facecolor="#111a2b", edgecolor="#4b5563", lw=0.8))
for q in ["BL", "BQ", "DQ", "DL"]:
    m = quad == q
    ax.add_collection(PolyCollection(list(hex_rings(cen[m])), facecolor=COL[q], edgecolor=COL[q], lw=0.1,
                                     alpha=0.9 if q in ("DQ", "DL") else 0.55))
for (x, y), n in zip(zip(sp.geometry.x, sp.geometry.y), site_noise):
    ax.plot(x, y, marker="o" if n >= LOUD_CUT else "x", ms=4.2 if n >= LOUD_CUT else 6.2,
            color="#f8fafc" if n >= LOUD_CUT else "#fca5a5", zorder=6)
bx, by, bx1, by1 = TH.bounds
ax.set_xlim(bx - 60_000, bx1 + 60_000); ax.set_ylim(by - 150_000, by1 + 40_000)
ax.set_title("Day 11 · Senses: Sound — ที่ที่มืดที่สุด คือที่ที่เงียบที่สุดไหม" + chr(10) +
             f"สองแกนเป็น proxy ทั้งคู่: ระยะถึงจุดทะเบียน (มืด) × ระยะถึงแหล่งเสียงที่ใกล้ที่สุด (เงียบ)",
             color=INK, fontsize=12.5, pad=26, fontweight="bold")
ax.text(0.5, 1.012, f"Spearman(ระยะถึงจุดทะเบียน, ระยะถึงแหล่งเสียง) = {rho:+.3f} · "
        f"เกณฑ์: มืด = ≤{DARK_CUT:.0f} กม. จากจุดทะเบียน · เงียบ = ≥10 กม. จากแหล่งเสียง",
        transform=ax.transAxes, ha="center", color="#94a3b8", fontsize=8.6)
# legend 2 แถว เว้นระยะคำนวณจากความยาวข้อความ (กันข้อความทับกันแบบรอบแรก)
LEG = [(-0.02, 0.0, "DQ", f"มืด + เงียบ ({cnt['DQ']})"), (0.30, 0.0, "DL", f"มืด แต่มีเสียงรบกวน ({cnt['DL']})"),
       (0.62, 0.0, "BQ", f"สว่าง + เงียบ ({cnt['BQ']})"), (-0.02, -0.045, "BL", f"สว่าง + มีเสียง ({cnt['BL']})"),
       (0.30, -0.045, "site_quiet", f"จุดทะเบียนที่เงียบ ≥{LOUD_CUT:.0f} กม."),
       (0.62, -0.045, "site_loud", f"จุดทะเบียนที่เสียงถึง <{LOUD_CUT:.0f} กม. ({loud_sites} แห่ง)")]
for xf, yf, key, lab in LEG:
    if key in COL:
        ax.plot([xf], [yf], marker="s", color=COL[key], ms=9, clip_on=False, transform=ax.transAxes, zorder=9)
    elif key == "site_quiet":
        ax.plot([xf], [yf], marker="o", color="#f8fafc", ms=5.5, clip_on=False, transform=ax.transAxes, zorder=9)
    else:
        ax.plot([xf], [yf], marker="x", color="#fca5a5", ms=7, mew=1.6, clip_on=False, transform=ax.transAxes, zorder=9)
    ax.text(xf + 0.022, yf, lab, transform=ax.transAxes, fontsize=7.6, color="#cbd5e1", va="center")

fig.savefig(os.path.join(OUT, "day11_sound_preview.png"), dpi=150, facecolor=BG, bbox_inches="tight")
log("เขียน day11_sound_preview.png")

### 6. scatter + bar

In [ ]:
fig2, (a1, a2) = plt.subplots(1, 2, figsize=(13.6, 5.6), facecolor=BG, gridspec_kw=dict(width_ratios=[1.15, 1]))
for a in (a1, a2): a.set_facecolor("#111a2b")
for q in ["BL", "BQ", "DQ", "DL"]:
    m = quad == q
    a1.scatter(km_dark[m], noise_km[m], s=9, color=COL[q], alpha=0.8, label=f"{LAB[q]} ({int(m.sum())})")
a1.axvline(DARK_CUT, color="#94a3b8", ls="--", lw=1.0)
a1.axhline(10.0, color="#94a3b8", ls="--", lw=1.0)
a1.set_xlabel("km_nearest — ระยะจาก hex ถึงจุดทะเบียนที่ใกล้สุด (กม.)", color="#cbd5e1")
a1.set_ylabel("ระยะจาก hex ถึงแหล่งเสียงที่ใกล้สุด (กม.)", color="#cbd5e1")
a1.set_title(f"hex 1,738 เซลล์ — มืดกับเงียบไปด้วยกันไหม (Spearman {rho:+.3f})", color=INK, fontsize=11)
a1.legend(frameon=False, labelcolor="#cbd5e1", fontsize=8, loc="upper right")
order = np.argsort(site_noise)
show = np.concatenate([order[:15], order[-15:]])       # 15 จุดที่เสียงใกล้ที่สุด + 15 จุดที่ไกลที่สุด
a2.barh(range(len(show)), site_noise[show],
        color=["#f4566a" if site_noise[i2] < LOUD_CUT else "#2dd4bf" for i2 in show])
a2.set_yticks(range(len(show)), [str(df.name.iloc[i2])[:26] for i2 in show], fontsize=7.2)
a2.invert_yaxis()
a2.set_xlabel("ระยะจากจุดทะเบียนถึงแหล่งเสียงที่ใกล้ที่สุด (กม.)", color="#cbd5e1")
a2.set_title(f"15 จุดที่เสียงใกล้ที่สุด (บน) และ 15 จุดที่ไกลที่สุด (ล่าง)" + chr(10) +
             f"{loud_sites} จาก 83 แห่งอยู่ห่างแหล่งเสียง <{LOUD_CUT:.0f} กม. · มัธยฐาน {np.median(site_noise):.1f} กม.",
             color=INK, fontsize=10.5)
a2.text(0.98, 0.05, "แหล่งเสียงที่ใกล้จุดทะเบียนที่สุด: " +
        " · ".join(f"{k} {v}" for k, v in site_src.items()),
        transform=a2.transAxes, ha="right", fontsize=7.4, color="#94a3b8")

for a in (a1, a2):
    for sp_ in a.spines.values(): sp_.set_color("#4b5563")
    a.tick_params(colors="#94a3b8"); a.grid(alpha=0.12, color="#4b5563")
a2.tick_params(axis="y", labelsize=7.2)
fig2.tight_layout(); fig2.savefig(os.path.join(OUT, "day11_compare.png"), dpi=150, facecolor=BG)
log("เขียน day11_compare.png")

### 7. stats

In [ ]:
area_km2 = cen.area / 1e6
src_cnt = {NAMES[i]: int((which == i).sum()) for i in range(4)}
site_src = {NAMES[i]: int((site_which == i).sum()) for i in range(4)}
stat = {
    "hex_total": int(len(km_dark)), "site_total": int(len(df)),
    "dark_cut_km": DARK_CUT, "quiet_cut_km": 10.0, "loud_site_cut_km": LOUD_CUT,
    "metric_definitions": {
        "km_nearest": "per hex = ระยะถึงจุดอนุรักษ์ท้องฟ้ามืดในทะเบียนที่ใกล้ที่สุด (proxy ของความมืด)",
        "noise_km": "per hex = min ระยะถึงแหล่งเสียง 4 ชนิด (ถนนสายหลัก/รถไฟ/พื้นที่เมือง/สนามบิน)",
        "quadrant": "จัดกลุ่ม hex ด้วยเกณฑ์ dark ≤25 กม. และ quiet ≥10 กม.",
        "spearman": "สหสัมพันธ์ระดับ hex ระหว่าง km_nearest กับ noise_km",
        "site_noise_km": "per registered site = min ระยะถึงแหล่งเสียงทั้ง 4 ชนิด"},
    "darkness_km": {"mean": round(float(km_dark.mean()), 2), "median": round(float(np.median(km_dark)), 2),
                    "min": round(float(km_dark.min()), 2), "max": round(float(km_dark.max()), 1)},
    "noise_km": {"mean": round(float(noise_km.mean()), 2), "median": round(float(np.median(noise_km)), 2),
                 "min": round(float(noise_km.min()), 2), "max": round(float(noise_km.max()), 1)},
    "quadrants": {q: {"hex": cnt[q], "area_km2": int(area_km2[quad == q].sum())} for q in ["DQ", "DL", "BQ", "BL"]},
    "spearman_dark_vs_noise": round(float(rho), 3), "spearman_p": float(pv),
    "sensitivity_major_highway_only": {"spearman": round(float(rho_m), 3),
                                       "quiet_hex_count_major": int((noise_major >= 10).sum()),
                                       "road_lines_used": {"all": int(len(road_all)), "major": int(len(road_major))}},
    "nearest_noise_source_by_hex": src_cnt, "nearest_noise_source_by_site": site_src,
    "sites_loud_under_5km": loud_sites,
    "site_noise_km_stats": {"median": round(float(np.median(site_noise)), 1),
                            "min": round(float(site_noise.min()), 2), "max": round(float(site_noise.max()), 1)},
    "quietest_dark_sites": [{"name": str(df.name.iloc[order[-i]]), "noise_km": round(float(site_noise[order[-i]]), 1)}
                            for i in range(1, 6)],
    "loudest_dark_sites": [{"name": str(df.name.iloc[order[i]]), "noise_km": round(float(site_noise[order[i]]), 2),
                            "source": NAMES[int(site_which[order[i]])]} for i in range(5)],
    "note": "ทั้งสองแกนเป็น proxy จากข้อมูล Natural Earth 10m (ถนน/รถไฟ/เมือง/สนามบิน) "
            "ไม่ใช่ค่าวัดระดับเสียงจริง (dB) และไม่ใช่ค่า sky brightness จริง"}
json.dump(stat, open(os.path.join(OUT, "day11_stats.json"), "w"), ensure_ascii=False, indent=1)
log(f"แหล่งเสียงที่ใกล้ hex ที่สุด: {src_cnt}")
log(f"แหล่งเสียงที่ใกล้จุดทะเบียนที่สุด (เสียงถึง <5 กม. = {loud_sites} แห่ง): {site_src}")
from shapely.geometry import mapping as _mapping
gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature",
     "properties": {"km_nearest": float(km_dark[k]), "noise_km": round(float(noise_km[k]), 2),
                    "nearest_source": NAMES[int(which[k])], "quadrant": str(quad[k])},
     "geometry": _mapping(h)} for k, h in enumerate(hexes)]}
json.dump(gj, open(os.path.join(OUT, "day11_hex_quadrants.geojson"), "w"), ensure_ascii=False)
log("เขียน day11_hex_quadrants.geojson")
log(f"quietest sites: {[s['name'] for s in stat['quietest_dark_sites']]}")
log(f"loudest sites : {[(s['name'], s['noise_km'], s['source']) for s in stat['loudest_dark_sites']]}")
log(f"รวมเวลา {time.time()-T0:.1f} วิ")